# TurboQuant Algorithm Tests

Tests covering every layer of the pipeline:
1. Rotation matrix correctness
2. Codebook (Lloyd-Max) sanity
3. Algorithm 1 — TurboQuantMSE
4. Algorithm 2 — TurboQuantProd (unbiasedness)
5. Value quantization
6. RingBuffer
7. KVCaptureEngine
8. CompressedKVStore
9. Hybrid attention
10. End-to-end pipeline

In [2]:
import sys
sys.path.insert(0, '..')

import torch
import math
import numpy as np

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

def check(name, cond, detail=''):
    status = 'PASS' if cond else 'FAIL'
    print(f'  [{status}]  {name}' + (f'  ({detail})' if detail else ''))
    return cond

Device: cpu


## 1. Rotation Matrix

In [3]:
from turboquant.rotation import generate_rotation_matrix, generate_qjl_matrix, rotate_forward, rotate_backward

print('=== Rotation Matrix ===')
for d in [64, 128]:
    Pi = generate_rotation_matrix(d, DEVICE)

    # Orthogonality: Pi.T @ Pi == I
    ortho_err = (Pi.T @ Pi - torch.eye(d, device=DEVICE)).abs().max().item()
    check(f'd={d} orthogonality error < 1e-5', ortho_err < 1e-5, f'{ortho_err:.2e}')

    # Determinism
    Pi2 = generate_rotation_matrix(d, DEVICE, seed=42)
    check(f'd={d} deterministic (same seed)', torch.allclose(Pi, Pi2))

    # Different seeds differ
    Pi3 = generate_rotation_matrix(d, DEVICE, seed=99)
    check(f'd={d} different seeds produce different matrices', not torch.allclose(Pi, Pi3))

    # Forward then backward == identity
    x = torch.randn(50, d, device=DEVICE)
    err = (rotate_backward(rotate_forward(x, Pi), Pi) - x).abs().max().item()
    check(f'd={d} forward->backward roundtrip', err < 1e-5, f'{err:.2e}')

=== Rotation Matrix ===
  [PASS]  d=64 orthogonality error < 1e-5  (4.77e-07)
  [PASS]  d=64 deterministic (same seed)
  [PASS]  d=64 different seeds produce different matrices
  [PASS]  d=64 forward->backward roundtrip  (1.91e-06)
  [PASS]  d=128 orthogonality error < 1e-5  (5.96e-07)
  [PASS]  d=128 deterministic (same seed)
  [PASS]  d=128 different seeds produce different matrices
  [PASS]  d=128 forward->backward roundtrip  (2.38e-06)


## 2. Codebook (Lloyd-Max)

In [5]:
from turboquant.codebook import get_codebook, get_codebook_tensors
import numpy as np

print('=== Codebook ===')
for d, bits in [(64, 2), (768, 3), (128, 4)]:
    cb = get_codebook(d, bits)
    centroids  = np.array(cb['centroids'])
    boundaries = np.array(cb['boundaries'])

    check(f'd={d} b={bits} centroid count == 2^bits', len(centroids) == 2**bits, f'{len(centroids)}')
    check(f'd={d} b={bits} boundary count == 2^bits+1', len(boundaries) == 2**bits + 1)
    check(f'd={d} b={bits} boundaries span [-1,1]',
          abs(boundaries[0] + 1.0) < 1e-9 and abs(boundaries[-1] - 1.0) < 1e-9)
    check(f'd={d} b={bits} centroids sorted', all(centroids[i] < centroids[i+1] for i in range(len(centroids)-1)))
    check(f'd={d} b={bits} centroids inside [-1,1]', centroids.min() > -1 and centroids.max() < 1)
    all_inside = all(boundaries[i] < centroids[i] < boundaries[i+1] for i in range(len(centroids)))
    check(f'd={d} b={bits} each centroid inside its bin', all_inside)

    c_t, b_t = get_codebook_tensors(d, bits, DEVICE)
    check(f'd={d} b={bits} GPU tensor shapes', c_t.shape == (2**bits,) and b_t.shape == (2**bits+1,))

=== Codebook ===
  [PASS]  d=64 b=2 centroid count == 2^bits  (4)
  [PASS]  d=64 b=2 boundary count == 2^bits+1
  [PASS]  d=64 b=2 boundaries span [-1,1]
  [PASS]  d=64 b=2 centroids sorted
  [PASS]  d=64 b=2 centroids inside [-1,1]
  [PASS]  d=64 b=2 each centroid inside its bin
  [PASS]  d=64 b=2 GPU tensor shapes
[TurboQuant] Computing Lloyd-Max codebook for d=768, bits=3...
[TurboQuant] MSE per coord = 4.485719e-05, total MSE = 0.034450
  [PASS]  d=768 b=3 centroid count == 2^bits  (8)
  [PASS]  d=768 b=3 boundary count == 2^bits+1
  [PASS]  d=768 b=3 boundaries span [-1,1]
  [PASS]  d=768 b=3 centroids sorted
  [PASS]  d=768 b=3 centroids inside [-1,1]
  [PASS]  d=768 b=3 each centroid inside its bin
  [PASS]  d=768 b=3 GPU tensor shapes
  [PASS]  d=128 b=4 centroid count == 2^bits  (16)
  [PASS]  d=128 b=4 boundary count == 2^bits+1
  [PASS]  d=128 b=4 boundaries span [-1,1]
  [PASS]  d=128 b=4 centroids sorted
  [PASS]  d=128 b=4 centroids inside [-1,1]
  [PASS]  d=128 b=4 each 

## 3. Algorithm 1 — TurboQuantMSE

In [12]:
from turboquant.quantizer import TurboQuantMSE
import torch.nn.functional as F

print('=== TurboQuantMSE (Algorithm 1) ===')
for bits in [2, 3, 4]:
    dim = 128
    q = TurboQuantMSE(dim=dim, bits=bits, device=DEVICE)
    x = torch.randn(200, dim, device=DEVICE)

    mse_q = q.quantize(x)
    x_hat = q.dequantize(mse_q)

    check(f'b={bits} dequant shape preserved', x_hat.shape == x.shape)

    norm_err = (mse_q.norms.to(DEVICE) - x.norm(dim=-1)).abs().max().item()
    check(f'b={bits} norms stored correctly', norm_err < 1e-4, f'{norm_err:.2e}')

    cos_sim = F.cosine_similarity(x, x_hat).mean().item()
    thresh = {2: 0.85, 3: 0.95, 4: 0.99}[bits]
    check(f'b={bits} cosine sim > {thresh}', cos_sim > thresh, f'{cos_sim:.4f}')

    norm_ratio = (x_hat.norm(dim=-1) / x.norm(dim=-1)).mean().item()
    check(f'b={bits} norm ratio ~= 1.0', abs(norm_ratio - 1.0) < 0.05, f'{norm_ratio:.4f}')

print()
q128 = TurboQuantMSE(dim=128, bits=3, device=DEVICE)

x_batch = torch.randn(4, 8, 128, device=DEVICE)
check('batched input (4,8,128) shape preserved', q128(x_batch).shape == x_batch.shape)

x_zero = torch.zeros(5, 128, device=DEVICE)
check('zero vectors produce no NaN', not torch.isnan(q128(x_zero)).any())

print('\nMSE vs bits:')
x_ref = torch.randn(1000, 128, device=DEVICE)
for bits in [1, 2, 3, 4]:
    qq = TurboQuantMSE(dim=128, bits=bits, device=DEVICE)
    mse = ((x_ref - qq(x_ref))**2).mean().item()
    print(f'  {bits}-bit MSE: {mse:.4f}')

=== TurboQuantMSE (Algorithm 1) ===
  [PASS]  b=2 dequant shape preserved
  [PASS]  b=2 norms stored correctly  (0.00e+00)
  [PASS]  b=2 cosine sim > 0.85  (0.9401)
  [FAIL]  b=2 norm ratio ~= 1.0  (0.9410)
  [PASS]  b=3 dequant shape preserved
  [PASS]  b=3 norms stored correctly  (0.00e+00)
  [PASS]  b=3 cosine sim > 0.95  (0.9834)
  [PASS]  b=3 norm ratio ~= 1.0  (0.9837)
  [PASS]  b=4 dequant shape preserved
  [PASS]  b=4 norms stored correctly  (0.00e+00)
  [PASS]  b=4 cosine sim > 0.99  (0.9954)
  [PASS]  b=4 norm ratio ~= 1.0  (0.9954)

  [PASS]  batched input (4,8,128) shape preserved
  [PASS]  zero vectors produce no NaN

MSE vs bits:
  1-bit MSE: 0.3624
  2-bit MSE: 0.1168
  3-bit MSE: 0.0344
  4-bit MSE: 0.0094


## 4. Algorithm 2 — TurboQuantProd (Unbiasedness)

In [20]:
from turboquant.quantizer import TurboQuantProd
import torch.nn.functional as F

print('=== TurboQuantProd (Algorithm 2) ===')
dim = 128

for bits in [2, 3, 4]:
    tqp = TurboQuantProd(dim=dim, bits=bits, device=DEVICE)
    x = torch.randn(500, dim, device=DEVICE)
    x_hat = tqp(x)

    check(f'b={bits} shape preserved', x_hat.shape == x.shape)
    cos_sim = F.cosine_similarity(x, x_hat).mean().item()
    thresh = {2: 0.80, 3: 0.90, 4: 0.97}[bits]
    check(f'b={bits} cosine sim > {thresh}', cos_sim > thresh, f'{cos_sim:.4f}')

print()
print('--- Unbiasedness of inner product estimator ---')
torch.manual_seed(0)
tqp3 = TurboQuantProd(dim=dim, bits=3, device=DEVICE)
keys    = torch.randn(200, dim, device=DEVICE)
queries = torch.randn(200, dim, device=DEVICE)

true_scores = (queries * keys).sum(-1)
keys_hat    = tqp3.dequantize(tqp3.quantize(keys))
est_scores  = (queries * keys_hat).sum(-1)

bias    = (est_scores - true_scores).mean().item()
rel_err = ((est_scores - true_scores).abs() / (true_scores.abs() + 1e-6)).mean().item()
corr    = torch.corrcoef(torch.stack([true_scores, est_scores]))[0, 1].item()

check('bias ~= 0  (|bias| < 0.5)', abs(bias) < 0.5, f'{bias:.4f}')
check('relative error < 30%', rel_err < 0.30, f'{rel_err*100:.1f}%')
check('Pearson corr > 0.90', corr > 0.90, f'{corr:.4f}')

print(f'  True mean={true_scores.mean().item():.4f}  Est mean={est_scores.mean().item():.4f}')
print(f'  Pearson corr = {corr:.4f}')

=== TurboQuantProd (Algorithm 2) ===
  [PASS]  b=2 shape preserved
  [FAIL]  b=2 cosine sim > 0.8  (0.7978)
  [PASS]  b=3 shape preserved
  [PASS]  b=3 cosine sim > 0.9  (0.9198)
  [PASS]  b=4 shape preserved
  [PASS]  b=4 cosine sim > 0.97  (0.9740)

--- Unbiasedness of inner product estimator ---
  [PASS]  bias ~= 0  (|bias| < 0.5)  (-0.0977)
  [FAIL]  relative error < 30%  (226.2%)
  [PASS]  Pearson corr > 0.90  (0.9246)
  True mean=-0.0991  Est mean=-0.1968
  Pearson corr = 0.9246


In [21]:
# attention_score() and dequantize+matmul must be numerically equivalent
print('--- attention_score == dequantize+matmul ---')
tqp = TurboQuantProd(dim=64, bits=3, device=DEVICE)
K = torch.randn(32, 64, device=DEVICE)
Q = torch.randn(4,  64, device=DEVICE)

# Method A: dequantize then matmul
scores_a = Q @ tqp.dequantize(tqp.quantize(K)).T   # (4, 32)

# Method B: attention_score (expects (..., n_q, d) and ProdQuantized of (..., n_k, ...))
pq_3d    = tqp.quantize(K.unsqueeze(0))             # (1, 32, ...)
scores_b = tqp.attention_score(Q.unsqueeze(0), pq_3d).squeeze(0)  # (4, 32)

max_diff = (scores_a - scores_b).abs().max().item()
check('max diff < 1e-3', max_diff < 1e-3, f'{max_diff:.2e}')

--- attention_score == dequantize+matmul ---
  [PASS]  max diff < 1e-3  (3.81e-06)


True

## 5. Value Quantization

In [31]:
from turboquant.kv_cache import quantize_values, dequantize_values, unpack_values
import torch.nn.functional as F

print('=== Value Quantization ===')
for bits in [2, 4]:
    v  = torch.randn(8, 4, 128, device=DEVICE)
    vq = quantize_values(v, bits=bits, group_size=32)

    exp_packed = 128 // (8 // bits)   # 2-bit: 32 bytes, 4-bit: 64 bytes
    check(f'b={bits} packed dim == {exp_packed}', vq.data.shape[-1] == exp_packed, f'{vq.data.shape[-1]}')
    check(f'b={bits} n_groups == 4', vq.scales.shape[-1] == 4)

    v_hat = dequantize_values(vq, group_size=32)
    check(f'b={bits} dequant shape preserved', v_hat.shape == v.shape)
    check(f'b={bits} no NaN', not torch.isnan(v_hat).any())
    check(f'b={bits} no Inf', not torch.isinf(v_hat).any())

    cos_sim = F.cosine_similarity(v.reshape(-1,128), v_hat.reshape(-1,128)).mean().item()
    thresh = {2: 0.90, 4: 0.999}[bits]
    check(f'b={bits} cosine sim > {thresh}', cos_sim > thresh, f'{cos_sim:.4f}')

print()
print('--- Pack/unpack roundtrip ---')
for bits in [2, 4]:
    v  = torch.randn(4, 128, device=DEVICE)
    vq = quantize_values(v, bits=bits, group_size=32)
    up = unpack_values(vq)
    check(f'b={bits} unpacked values in [0, {2**bits-1}]',
          up.min().item() >= 0 and up.max().item() < 2**bits,
          f'range [{up.min().item()}, {up.max().item()}]')

=== Value Quantization ===
  [PASS]  b=2 packed dim == 32  (32)
  [PASS]  b=2 n_groups == 4
  [PASS]  b=2 dequant shape preserved
  [PASS]  b=2 no NaN
  [PASS]  b=2 no Inf
  [PASS]  b=2 cosine sim > 0.9  (0.9331)
  [PASS]  b=4 packed dim == 64  (64)
  [PASS]  b=4 n_groups == 4
  [PASS]  b=4 dequant shape preserved
  [PASS]  b=4 no NaN
  [PASS]  b=4 no Inf
  [FAIL]  b=4 cosine sim > 0.999  (0.9970)

--- Pack/unpack roundtrip ---
  [PASS]  b=2 unpacked values in [0, 3]  (range [0, 3])
  [PASS]  b=4 unpacked values in [0, 15]  (range [0, 15])


## 6. RingBuffer

In [24]:
from turboquant.capture import RingBuffer

print('=== RingBuffer ===')
H, D, CAP = 4, 64, 32
ring = RingBuffer(capacity=CAP, num_kv_heads=H, head_dim=D, device=DEVICE)

# FIX: Add dtype=torch.bfloat16
k10 = torch.randn(10, H, D, device=DEVICE, dtype=torch.bfloat16)
v10 = torch.randn(10, H, D, device=DEVICE, dtype=torch.bfloat16)
overflow = ring.write(k10, v10, 10)
check('write 10 < CAP: no overflow', overflow is None)
check('size == 10', ring.size == 10)

pk, pv = ring.peek()
check('peek returns written data', torch.allclose(pk, k10) and torch.allclose(pv, v10))

# Fill to capacity
k22 = torch.randn(22, H, D, device=DEVICE)  # This one is fine since it's just meant to overflow
overflow = ring.write(k22, torch.randn(22, H, D, device=DEVICE), 22)
check('write 22 more: fills to CAP, no overflow', overflow is None)
check('size == CAP', ring.size == CAP)

# Write 1 more: overflow of full buffer
k1 = torch.randn(1, H, D, device=DEVICE)
overflow = ring.write(k1, torch.randn(1, H, D, device=DEVICE), 1)
check('1 more: overflow returned', overflow is not None)
check('overflow len == CAP', overflow[0].shape[0] == CAP)
check('ring size == 1 after overflow', ring.size == 1)

dk, dv = ring.drain()
check('drain returns 1 token', dk.shape[0] == 1)
check('size == 0 after drain', ring.size == 0)
check('drain on empty == None', ring.drain() is None)

ring.write(k10, v10, 10)
ring.reset()
check('reset clears size', ring.size == 0)
check('reset clears total_written', ring.total_written == 0)


=== RingBuffer ===
  [PASS]  write 10 < CAP: no overflow
  [PASS]  size == 10
  [PASS]  peek returns written data
  [PASS]  write 22 more: fills to CAP, no overflow
  [PASS]  size == CAP
  [PASS]  1 more: overflow returned
  [PASS]  overflow len == CAP
  [PASS]  ring size == 1 after overflow
  [PASS]  drain returns 1 token
  [PASS]  size == 0 after drain
  [PASS]  drain on empty == None
  [PASS]  reset clears size
  [PASS]  reset clears total_written


True

## 7. KVCaptureEngine

In [25]:
from turboquant.capture import KVCaptureEngine
from turboquant.store import CompressedKVStore

print('=== KVCaptureEngine ===')
H, D = 2, 64
store  = CompressedKVStore(head_dim=D, num_kv_heads=H, key_bits=3, value_bits=2, device=DEVICE)
engine = KVCaptureEngine(store, ring_capacity=32, device=DEVICE)

# Short prefill stays in ring
engine.ingest_prefill(torch.randn(20, H, D, device=DEVICE), torch.randn(20, H, D, device=DEVICE), 20)
check('short prefill: 20 buffered', engine.total_buffered_tokens == 20)
check('short prefill: 0 compressed', engine.total_compressed_tokens == 0)

# Long prefill: overflow goes to store
engine.reset()
engine.ingest_prefill(torch.randn(100, H, D, device=DEVICE), torch.randn(100, H, D, device=DEVICE), 100)
check('long prefill: ring == 32', engine.total_buffered_tokens == 32)
check('long prefill: compressed == 68', engine.total_compressed_tokens == 68)
check('long prefill: total == 100', engine.total_tokens == 100)

# Decode tokens
for _ in range(5):
    engine.ingest_decode(torch.randn(1, H, D, device=DEVICE), torch.randn(1, H, D, device=DEVICE), 1)
check('5 decode steps: total == 105', engine.total_tokens == 105)

# Flush
before = engine.total_compressed_tokens
engine.flush()
check('flush: compressed increases', engine.total_compressed_tokens > before)
check('flush: ring is empty', engine.total_buffered_tokens == 0)

=== KVCaptureEngine ===
  [PASS]  short prefill: 20 buffered
  [PASS]  short prefill: 0 compressed
  [PASS]  long prefill: ring == 32
  [PASS]  long prefill: compressed == 68
  [PASS]  long prefill: total == 100
  [PASS]  5 decode steps: total == 105
  [PASS]  flush: compressed increases
  [PASS]  flush: ring is empty


True

## 8. CompressedKVStore

In [26]:
from turboquant.store import CompressedKVStore

print('=== CompressedKVStore ===')
H, D = 4, 128
store = CompressedKVStore(head_dim=D, num_kv_heads=H, key_bits=3, value_bits=2, device=DEVICE)

check('empty: get_flat_cache == None', store.get_flat_cache() is None)
check('empty: num_tokens == 0', store.num_tokens == 0)

store.append_chunk(torch.randn(32, H, D, device=DEVICE), torch.randn(32, H, D, device=DEVICE))
check('1 chunk: num_tokens == 32', store.num_tokens == 32)
check('1 chunk: num_chunks == 1', store.num_chunks == 1)

flat = store.get_flat_cache()
check('flat not None', flat is not None)
check('flat.num_tokens == 32', flat.num_tokens == 32)
check('flat is cached (same object)', store.get_flat_cache() is flat)

store.append_chunk(torch.randn(16, H, D, device=DEVICE), torch.randn(16, H, D, device=DEVICE))
flat2 = store.get_flat_cache()
check('cache invalidated after append', flat2 is not flat)
check('combined num_tokens == 48', flat2.num_tokens == 48)
check('memory_bytes > 0', store.memory_bytes() > 0)

store.reset()
check('reset: num_tokens == 0', store.num_tokens == 0)
check('reset: flat is None', store.get_flat_cache() is None)

=== CompressedKVStore ===
  [PASS]  empty: get_flat_cache == None
  [PASS]  empty: num_tokens == 0
  [PASS]  1 chunk: num_tokens == 32
  [PASS]  1 chunk: num_chunks == 1
  [PASS]  flat not None
  [PASS]  flat.num_tokens == 32
  [PASS]  flat is cached (same object)
  [PASS]  cache invalidated after append
  [PASS]  combined num_tokens == 48
  [PASS]  memory_bytes > 0
  [PASS]  reset: num_tokens == 0
  [PASS]  reset: flat is None


True

## 9. Hybrid Attention

In [27]:
from turboquant.score import compute_hybrid_attention
from turboquant.store import CompressedKVStore

print('=== Hybrid Attention ===')
H_kv, H_q, D = 2, 4, 64   # GQA ratio = 2
store = CompressedKVStore(head_dim=D, num_kv_heads=H_kv, key_bits=3, value_bits=2,
                          value_group_size=32, device=DEVICE)
query    = torch.randn(1, H_q, D, device=DEVICE)
recent_k = torch.randn(10, H_kv, D, device=DEVICE)
recent_v = torch.randn(10, H_kv, D, device=DEVICE)

# Case 1: exact-only (no compressed history)
out = compute_hybrid_attention(query, store, recent_k, recent_v, H_q)
check('exact-only: shape (1, H_q, D)', out.shape == (1, H_q, D))
check('exact-only: no NaN', not torch.isnan(out).any())

# Case 2: compressed-only (>= 16 tokens)
store.append_chunk(torch.randn(32, H_kv, D, device=DEVICE), torch.randn(32, H_kv, D, device=DEVICE))
out2 = compute_hybrid_attention(query, store, None, None, H_q)
check('compressed-only: shape', out2.shape == (1, H_q, D))
check('compressed-only: no NaN', not torch.isnan(out2).any())

# Case 3: hybrid
out3 = compute_hybrid_attention(query, store, recent_k, recent_v, H_q)
check('hybrid: shape', out3.shape == (1, H_q, D))
check('hybrid: no NaN', not torch.isnan(out3).any())

# Case 4: empty store + no recent -> zeros
empty_store = CompressedKVStore(head_dim=D, num_kv_heads=H_kv, device=DEVICE)
out_zero = compute_hybrid_attention(query, empty_store, None, None, H_q)
check('empty inputs: output is zero', out_zero.abs().max().item() == 0.0)

# Case 5: fewer than MIN_HISTORY_FOR_TQ (16) compressed tokens
small_store = CompressedKVStore(head_dim=D, num_kv_heads=H_kv, device=DEVICE)
small_store.append_chunk(torch.randn(8, H_kv, D, device=DEVICE), torch.randn(8, H_kv, D, device=DEVICE))
out_small = compute_hybrid_attention(query, small_store, recent_k, recent_v, H_q)
check('<16 compressed tokens falls back to exact path', out_small.shape == (1, H_q, D))

=== Hybrid Attention ===
  [PASS]  exact-only: shape (1, H_q, D)
  [PASS]  exact-only: no NaN
  [PASS]  compressed-only: shape
  [PASS]  compressed-only: no NaN
  [PASS]  hybrid: shape
  [PASS]  hybrid: no NaN
  [PASS]  empty inputs: output is zero
  [PASS]  <16 compressed tokens falls back to exact path


True

In [28]:
# Approximation quality: compressed attention vs exact
print('--- Approximation quality ---')
torch.manual_seed(42)
H_kv, H_q, D, N = 1, 1, 64, 128
scale = 1.0 / math.sqrt(D)

keys   = torch.randn(N, H_kv, D, device=DEVICE)
values = torch.randn(N, H_kv, D, device=DEVICE)
query  = torch.randn(1, H_q, D, device=DEVICE)

# Exact attention
k_flat = keys.squeeze(1)
v_flat = values.squeeze(1)
q_flat = query.squeeze()
w      = torch.softmax((k_flat @ q_flat) * scale, dim=0)
out_exact = (w.unsqueeze(-1) * v_flat).sum(0)

# TurboQuant compressed (4-bit for best quality)
qs = CompressedKVStore(head_dim=D, num_kv_heads=H_kv, key_bits=4, value_bits=4,
                       value_group_size=32, device=DEVICE)
qs.append_chunk(keys, values)
out_tq = compute_hybrid_attention(query, qs, None, None, H_q, scale=scale).squeeze()

import torch.nn.functional as F
cos_sim = F.cosine_similarity(out_exact.unsqueeze(0), out_tq.unsqueeze(0)).item()
check('4-bit compressed attn cosine sim > 0.9', cos_sim > 0.9, f'{cos_sim:.4f}')
print(f'  Cosine similarity (exact vs 4-bit TQ): {cos_sim:.4f}')

--- Approximation quality ---
  [FAIL]  4-bit compressed attn cosine sim > 0.9  (0.8270)
  Cosine similarity (exact vs 4-bit TQ): 0.8270


## 10. End-to-End Pipeline

In [29]:
from turboquant.capture import KVCaptureEngine
from turboquant.store import CompressedKVStore
from turboquant.score import compute_hybrid_attention

print('=== End-to-End Pipeline ===')
NUM_LAYERS   = 4
H_kv, H_q, D = 2, 4, 128
RING_CAP      = 32
PREFILL_LEN   = 200
DECODE_STEPS  = 50

stores  = [CompressedKVStore(head_dim=D, num_kv_heads=H_kv, key_bits=3, value_bits=2,
                              value_group_size=32, device=DEVICE, layer_idx=i)
           for i in range(NUM_LAYERS)]
engines = [KVCaptureEngine(stores[i], ring_capacity=RING_CAP, device=DEVICE)
           for i in range(NUM_LAYERS)]

# Prefill
torch.manual_seed(7)
for l in range(NUM_LAYERS):
    engines[l].ingest_prefill(
        torch.randn(PREFILL_LEN, H_kv, D, device=DEVICE),
        torch.randn(PREFILL_LEN, H_kv, D, device=DEVICE),
        PREFILL_LEN,
    )

check('prefill: each layer == 200 tokens',
      all(engines[l].total_tokens == PREFILL_LEN for l in range(NUM_LAYERS)))
check('prefill: ring == 32',
      all(engines[l].total_buffered_tokens == RING_CAP for l in range(NUM_LAYERS)))
check('prefill: 168 compressed',
      all(engines[l].total_compressed_tokens == PREFILL_LEN - RING_CAP for l in range(NUM_LAYERS)))

# Decode
all_ok = True
for step in range(DECODE_STEPS):
    query = torch.randn(1, H_q, D, device=DEVICE)
    for l in range(NUM_LAYERS):
        engines[l].ingest_decode(
            torch.randn(1, H_kv, D, device=DEVICE),
            torch.randn(1, H_kv, D, device=DEVICE), 1
        )
        rk, rv = engines[l].ring.peek() or (None, None)
        out = compute_hybrid_attention(query, stores[l], rk, rv, H_q)
        if torch.isnan(out).any():
            all_ok = False

check(f'{DECODE_STEPS} decode steps: no NaN', all_ok)
check(f'total tokens == {PREFILL_LEN + DECODE_STEPS}',
      all(engines[l].total_tokens == PREFILL_LEN + DECODE_STEPS for l in range(NUM_LAYERS)))

print()
print('Layer summary:')
for l in range(NUM_LAYERS):
    print(f'  Layer {l}: {engines[l].total_tokens} total | '
          f'{engines[l].total_compressed_tokens} compressed | '
          f'{engines[l].total_buffered_tokens} in ring | '
          f'{stores[l].memory_bytes():,} bytes')

=== End-to-End Pipeline ===
  [PASS]  prefill: each layer == 200 tokens
  [PASS]  prefill: ring == 32
  [PASS]  prefill: 168 compressed
  [PASS]  50 decode steps: no NaN
  [PASS]  total tokens == 250

Layer summary:
  Layer 0: 250 total | 232 compressed | 18 in ring | 46,400 bytes
  Layer 1: 250 total | 232 compressed | 18 in ring | 46,400 bytes
  Layer 2: 250 total | 232 compressed | 18 in ring | 46,400 bytes
  Layer 3: 250 total | 232 compressed | 18 in ring | 46,400 bytes


## 11. Known Issue: 3-bit Storage Bug

In [30]:
from turboquant.quantizer import _pack_indices, _unpack_indices

print('=== 3-bit packing audit ===')
print('bits  | expected bytes | actual bytes | note')
print('------+----------------+--------------+-----')
d = 128
for bits in [1, 2, 3, 4]:
    indices  = torch.randint(0, 2**bits, (d,))
    packed   = _pack_indices(indices, bits)
    stored_bits = bits if bits != 3 else 4   # 3-bit promoted to 4
    vpb      = 8 // stored_bits
    expected = math.ceil(d / vpb)
    actual   = packed.numel()
    note     = '  <-- BUG: stored as 4-bit' if bits == 3 else ''
    print(f'  {bits}   | {expected:14d} | {actual:12d} |{note}')

    unpacked = _unpack_indices(packed, bits, d)
    check(f'{bits}-bit roundtrip', torch.all(unpacked == indices.long()).item())

=== 3-bit packing audit ===
bits  | expected bytes | actual bytes | note
------+----------------+--------------+-----
  1   |             16 |           16 |
  [PASS]  1-bit roundtrip
  2   |             32 |           32 |
  [PASS]  2-bit roundtrip
  3   |             64 |           64 |  <-- BUG: stored as 4-bit
  [PASS]  3-bit roundtrip
  4   |             64 |           64 |
  [PASS]  4-bit roundtrip
